# Multi-Cycle Bottle Entry / Line-Capture / Exit Notebook

This notebook does the full workflow you asked for:

## Outputs
1. **Entrance frames**  
   Detect the first frame where a bottle pair enters using OpenCV in a **bottom-left ROI**.

2. **Full overlay video**  
   Create a full video overlay that shows:
   - `"WAIT FOR ENTRY"`
   - `"BOTTLE ENTERED"`
   - `"CAPTURE WITH LINES"`
   - `"BOTTLES EXITED"`

3. **Exit frames**  
   Save the frame where the two conditions agree (`&&`):
   - the **first / ahead / rightmost bottle** has reached or passed the **RIGHT line**
   - the **second / trailing / leftmost bottle** has reached or passed the **LEFT line**

---

## Important behavior implemented
- Each bottle center has its **own overlay color**.
- The **first/right bottle** changes color **as soon as it reaches/passes the RIGHT line**, independent of the second bottle.
- The **second/left bottle** changes color **as soon as it reaches/passes the LEFT line**, independent of the first bottle.
- When **both are passed** (`&&`), that frame is saved as the **exit frame** for that cycle.
- A video can contain **multiple cycles**:
  - entry
  - capture with lines
  - both passed
  - exited
  - then it waits for the next pair

---

## Assumption
Bottles move **from left to right**, so:
- **rightmost bottle** = first / ahead bottle
- **leftmost bottle** = second / trailing bottle

## 1. Imports

In [ ]:
from pathlib import Path
import math
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ultralytics import YOLO
except ImportError as exc:
    raise ImportError("Install ultralytics first: pip install ultralytics") from exc

plt.rcParams["figure.dpi"] = 120

## 2. Configuration

Set your paths and tune the line points here.

- `ENTRY_ROI_FRACTIONS` is the **bottom-left ROI** for first-entry detection.
- `DETECT_ROI_FRACTIONS` is the **bottom-half ROI** for NCNN bottle detection.
- By default the target class is:

```python
2: "Without_Cap"
```

Change it if you want another class.

In [ ]:
# -----------------------------
# Paths
# -----------------------------
VIDEO_PATH = Path("../data/5.avi")

BEST_PT_PATH = Path("../models/best.pt")
NCNN_MODEL_DIR = Path("../models/best_ncnn_model")

OUTPUT_DIR = Path("../outputs/multi_cycle_bottle_overlay")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OVERLAY_VIDEO_PATH = OUTPUT_DIR / "overlay_video.mp4"
SUMMARY_CSV_PATH = OUTPUT_DIR / "cycle_summary.csv"
EVENTS_CSV_PATH = OUTPUT_DIR / "event_log.csv"

ENTRANCE_FRAMES_DIR = OUTPUT_DIR / "entrance_frames"
EXIT_FRAMES_DIR = OUTPUT_DIR / "exit_frames"
ENTRANCE_FRAMES_DIR.mkdir(parents=True, exist_ok=True)
EXIT_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# OpenCV entry detection ROI (bottom-left)
# -----------------------------
ENTRY_ROI_FRACTIONS = {
    "x1": 0.00,
    "y1": 0.50,
    "x2": 0.45,
    "y2": 1.00,
}

BACKGROUND_WARMUP_FRAMES = 30
ENTRY_DIFF_THRESHOLD = 35
ENTRY_MORPH_KERNEL_SIZE = 7
ENTRY_MIN_FOREGROUND_PIXELS = 1200
ENTRY_MIN_LARGEST_CONTOUR_AREA = 800
ENTRY_CONSECUTIVE_FRAMES_REQUIRED = 2

# -----------------------------
# YOLO NCNN detect ROI (bottom half)
# -----------------------------
DETECT_ROI_FRACTIONS = {
    "x1": 0.00,
    "y1": 0.50,
    "x2": 1.00,
    "y2": 1.00,
}

YOLO_IMGSZ = 320
YOLO_CONF = 0.25
YOLO_IOU = 0.50

TARGET_CLASS_NAMES = ["Without_Cap"]
TARGET_CLASS_IDS = [2]  # set None if you only want matching by name

MAX_TARGETS = 2

# -----------------------------
# Fixed lines
# Replace these with your exact points.
# -----------------------------
LEFT_LINE_P1  = (273, 12)
LEFT_LINE_P2  = (374, 640)

RIGHT_LINE_P1 = (1117, 12)
RIGHT_LINE_P2 = (1010, 640)

LINE_TOLERANCE_PX = 8

# -----------------------------
# Cycle logic
# -----------------------------
NO_DETECTION_EXIT_FRAMES = 8   # after both passed, if detections disappear for this many frames => cycle ended
ENTRY_BANNER_FRAMES = 20
EXIT_BANNER_FRAMES = 20
PASS_BANNER_FRAMES = 20

# -----------------------------
# Colors (BGR)
# -----------------------------
COLOR_ENTRY_ROI = (255, 255, 0)
COLOR_DETECT_ROI = (255, 255, 255)

COLOR_LEFT_BEFORE  = (0, 255, 255)   # yellow
COLOR_RIGHT_BEFORE = (255, 0, 255)   # magenta
COLOR_PASSED       = (0, 255, 0)     # green
COLOR_LINE_LEFT    = (0, 255, 255)
COLOR_LINE_RIGHT   = (255, 0, 255)
COLOR_CENTER       = (0, 0, 255)

## 3. Utility functions

In [ ]:
def open_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")
    return cap


def get_video_info(video_path):
    cap = open_video(video_path)
    info = {
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    cap.release()
    if info["fps"] is None or info["fps"] <= 0:
        info["fps"] = 30.0
    return info


def roi_from_fractions(frame_shape, fractions):
    h, w = frame_shape[:2]
    x1 = int(round(fractions["x1"] * w))
    y1 = int(round(fractions["y1"] * h))
    x2 = int(round(fractions["x2"] * w))
    y2 = int(round(fractions["y2"] * h))
    x1 = max(0, min(w - 1, x1))
    y1 = max(0, min(h - 1, y1))
    x2 = max(x1 + 1, min(w, x2))
    y2 = max(y1 + 1, min(h, y2))
    return x1, y1, x2, y2


def crop_roi(frame, roi_rect):
    x1, y1, x2, y2 = roi_rect
    return frame[y1:y2, x1:x2]


def load_frame(video_path, frame_idx):
    cap = open_video(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def line_distance(point, p1, p2):
    px, py = point
    x1, y1 = p1
    x2, y2 = p2
    numerator = abs((y2 - y1) * px - (x2 - x1) * py + x2 * y1 - y2 * x1)
    denominator = math.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
    if denominator == 0:
        return float("inf")
    return numerator / denominator


def x_of_line_at_y(p1, p2, y):
    x1, y1 = p1
    x2, y2 = p2

    if y2 == y1:
        return (x1 + x2) / 2.0

    t = (y - y1) / (y2 - y1)
    return x1 + t * (x2 - x1)


def reached_or_passed_line(point, p1, p2, tolerance_px=8):
    # For left->right motion:
    # A bottle is considered to have reached/passed the line when its center
    # is either near the line OR to the right of the line at that y.
    px, py = point
    d = line_distance(point, p1, p2)
    x_line = x_of_line_at_y(p1, p2, py)
    on_or_right = px >= x_line
    return bool((d <= tolerance_px) or on_or_right), float(d), float(x_line)


def show_bgr(frame, title=None, figsize=(12, 7)):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


def draw_text_box(img, text, org, color=(255, 255, 255), bg=(0, 0, 0), scale=0.55, thickness=2):
    x, y = org
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)
    cv2.rectangle(img, (x, y - th - baseline - 6), (x + tw + 8, y + 6), bg, -1)
    cv2.putText(img, text, (x + 4, y - 4), font, scale, color, thickness, cv2.LINE_AA)


def add_roi_rectangle(img, roi_rect, label, color):
    x1, y1, x2, y2 = roi_rect
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    draw_text_box(img, label, (x1 + 5, y1 + 25), color=color, bg=(0, 0, 0), scale=0.55)


def draw_line_with_label(img, p1, p2, label, color, thickness=2):
    cv2.line(img, p1, p2, color, thickness)
    cv2.circle(img, p1, 4, color, -1)
    cv2.circle(img, p2, 4, color, -1)
    draw_text_box(img, label, (p1[0] + 8, max(p1[1] - 8, 20)), color=color, bg=(0, 0, 0), scale=0.55)


video_info = get_video_info(VIDEO_PATH)
print(json.dumps(video_info, indent=2))

## 4. Load / export NCNN model

In [ ]:
def ensure_ncnn_model(best_pt_path, ncnn_model_dir, imgsz=320):
    best_pt_path = Path(best_pt_path)
    ncnn_model_dir = Path(ncnn_model_dir)

    if ncnn_model_dir.exists():
        print("Using existing NCNN model:", ncnn_model_dir)
        return ncnn_model_dir

    if not best_pt_path.exists():
        raise FileNotFoundError(
            f"NCNN model not found and best.pt is missing:\n"
            f"NCNN dir: {ncnn_model_dir}\n"
            f"best.pt: {best_pt_path}"
        )

    print("NCNN model not found. Exporting from best.pt ...")
    model_pt = YOLO(str(best_pt_path))
    model_pt.export(format="ncnn", imgsz=imgsz)

    expected_dir = best_pt_path.with_name(best_pt_path.stem + "_ncnn_model")
    if expected_dir.exists():
        print("Exported NCNN model:", expected_dir)
        return expected_dir

    if ncnn_model_dir.exists():
        return ncnn_model_dir

    raise FileNotFoundError("NCNN export seems to have finished, but no NCNN directory was found.")


NCNN_MODEL_USED = ensure_ncnn_model(BEST_PT_PATH, NCNN_MODEL_DIR, imgsz=YOLO_IMGSZ)
model = YOLO(str(NCNN_MODEL_USED))

print("Loaded model:", NCNN_MODEL_USED)
print("Model names:", model.names)

## 5. OpenCV first-entry detection (bottom-left ROI)

In [ ]:
def build_entry_background(video_path, warmup_frames=30):
    cap = open_video(video_path)

    frames_gray = []
    first_frame = None
    entry_roi_rect = None

    for i in range(warmup_frames):
        ret, frame = cap.read()
        if not ret:
            break

        if first_frame is None:
            first_frame = frame.copy()
            entry_roi_rect = roi_from_fractions(frame.shape, ENTRY_ROI_FRACTIONS)

        roi = crop_roi(frame, entry_roi_rect)
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)
        frames_gray.append(gray)

    cap.release()

    if len(frames_gray) == 0:
        raise RuntimeError("No warmup frames captured for background.")

    background = np.median(np.stack(frames_gray, axis=0), axis=0).astype(np.uint8)
    return background, first_frame, entry_roi_rect


def entry_foreground(frame, background_gray, entry_roi_rect):
    roi = crop_roi(frame, entry_roi_rect)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    diff = cv2.absdiff(background_gray, gray)

    _, mask_raw = cv2.threshold(diff, ENTRY_DIFF_THRESHOLD, 255, cv2.THRESH_BINARY)

    kernel = np.ones((ENTRY_MORPH_KERNEL_SIZE, ENTRY_MORPH_KERNEL_SIZE), np.uint8)
    mask_clean = cv2.morphologyEx(mask_raw, cv2.MORPH_OPEN, kernel)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    fg_pixels = int(cv2.countNonZero(mask_clean))
    largest_area = 0.0
    largest_box = None

    if contours:
        c = max(contours, key=cv2.contourArea)
        largest_area = float(cv2.contourArea(c))
        largest_box = cv2.boundingRect(c)

    present = (
        fg_pixels >= ENTRY_MIN_FOREGROUND_PIXELS and
        largest_area >= ENTRY_MIN_LARGEST_CONTOUR_AREA
    )

    return {
        "roi": roi,
        "diff": diff,
        "mask_clean": mask_clean,
        "foreground_pixels": fg_pixels,
        "largest_area": largest_area,
        "largest_box": largest_box,
        "present": present,
    }


entry_background, first_frame, entry_roi_rect = build_entry_background(
    VIDEO_PATH,
    warmup_frames=BACKGROUND_WARMUP_FRAMES
)

print("Entry ROI:", entry_roi_rect)

## 6. YOLO NCNN detection on bottom-half ROI

In [ ]:
def resolve_target_class_ids(model, target_names=None, target_ids=None):
    ids = set()

    if target_ids is not None:
        for cid in target_ids:
            ids.add(int(cid))

    if target_names is not None:
        wanted = {str(n).lower() for n in target_names}
        for cid, name in model.names.items():
            if str(name).lower() in wanted:
                ids.add(int(cid))

    if not ids:
        print("WARNING: no target class IDs resolved; all classes will be used.")
        return None

    ids = sorted(ids)
    print("Target class IDs:", ids)
    print("Target class names:", [model.names[i] for i in ids if i in model.names])
    return ids


TARGET_CLASS_ID_LIST = resolve_target_class_ids(model, TARGET_CLASS_NAMES, TARGET_CLASS_IDS)


def detect_bottom_half(frame):
    detect_roi_rect = roi_from_fractions(frame.shape, DETECT_ROI_FRACTIONS)
    roi = crop_roi(frame, detect_roi_rect)
    rx1, ry1, _, _ = detect_roi_rect

    kwargs = {
        "imgsz": YOLO_IMGSZ,
        "conf": YOLO_CONF,
        "iou": YOLO_IOU,
        "verbose": False,
    }
    if TARGET_CLASS_ID_LIST is not None:
        kwargs["classes"] = TARGET_CLASS_ID_LIST

    results = model.predict(roi, **kwargs)

    rows = []
    if len(results) == 0 or results[0].boxes is None:
        return rows, detect_roi_rect, roi

    boxes = results[0].boxes
    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes.xyxy[i].detach().cpu().numpy().astype(float)
        conf = float(boxes.conf[i]) if boxes.conf is not None else np.nan
        cls_id = int(boxes.cls[i]) if boxes.cls is not None else -1
        cls_name = str(model.names.get(cls_id, cls_id))

        x1f = x1 + rx1
        y1f = y1 + ry1
        x2f = x2 + rx1
        y2f = y2 + ry1
        cx = (x1f + x2f) / 2.0
        cy = (y1f + y2f) / 2.0

        rows.append({
            "class_id": cls_id,
            "class_name": cls_name,
            "confidence": conf,
            "x1": float(x1f),
            "y1": float(y1f),
            "x2": float(x2f),
            "y2": float(y2f),
            "x_center": float(cx),
            "y_center": float(cy),
            "width": float(x2f - x1f),
            "height": float(y2f - y1f),
            "area": float((x2f - x1f) * (y2f - y1f)),
        })

    # keep top detections and sort left->right
    rows = sorted(rows, key=lambda r: r["area"], reverse=True)[:max(MAX_TARGETS, 2)]
    rows = sorted(rows, key=lambda r: r["x_center"])
    return rows, detect_roi_rect, roi


def choose_pair(rows):
    if len(rows) < 2:
        return None, None

    rows2 = sorted(rows, key=lambda r: r["area"], reverse=True)[:2]
    rows2 = sorted(rows2, key=lambda r: r["x_center"])

    left_bottle = rows2[0]   # second / trailing bottle
    right_bottle = rows2[1]  # first / ahead bottle

    return left_bottle, right_bottle

## 7. Overlay drawing

In [ ]:
def draw_one_bottle(img, row, role_label, target_line_name, passed_flag, target_line_points, before_color):
    x1 = int(round(row["x1"]))
    y1 = int(round(row["y1"]))
    x2 = int(round(row["x2"]))
    y2 = int(round(row["y2"]))
    cx = int(round(row["x_center"]))
    cy = int(round(row["y_center"]))

    point = (row["x_center"], row["y_center"])
    _, dist, _ = reached_or_passed_line(point, target_line_points[0], target_line_points[1], LINE_TOLERANCE_PX)

    color = COLOR_PASSED if passed_flag else before_color

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    cv2.circle(img, (cx, cy), 6, COLOR_CENTER, -1)
    cv2.circle(img, (cx, cy), 12, color, 2)

    status = "PASSED" if passed_flag else "WAIT"
    text1 = f"{role_label}: {row['class_name']} {row['confidence']:.2f}"
    text2 = f"{status} {target_line_name} | center=({cx},{cy}) | dist={dist:.1f}px"

    draw_text_box(img, text1, (x1, max(y1 - 30, 20)), color=color, bg=(0, 0, 0), scale=0.50)
    draw_text_box(img, text2, (x1, max(y1 - 8, 40)), color=color, bg=(0, 0, 0), scale=0.48)


def draw_overlay(frame, state, cycle_id, event_banner, event_countdown,
                 entry_roi_rect, detect_roi_rect,
                 left_bottle, right_bottle,
                 left_passed, right_passed,
                 left_dist=None, right_dist=None,
                 exact_both=False):
    out = frame.copy()

    add_roi_rectangle(out, entry_roi_rect, "ENTRY ROI (bottom-left)", COLOR_ENTRY_ROI)
    add_roi_rectangle(out, detect_roi_rect, "NCNN DETECT ROI (bottom-half)", COLOR_DETECT_ROI)

    draw_line_with_label(out, LEFT_LINE_P1, LEFT_LINE_P2, "LEFT line / second bottle", COLOR_LINE_LEFT)
    draw_line_with_label(out, RIGHT_LINE_P1, RIGHT_LINE_P2, "RIGHT line / first bottle", COLOR_LINE_RIGHT)

    if left_bottle is not None:
        draw_one_bottle(
            out, left_bottle,
            role_label="SECOND / LEFT bottle",
            target_line_name="LEFT line",
            passed_flag=left_passed,
            target_line_points=(LEFT_LINE_P1, LEFT_LINE_P2),
            before_color=COLOR_LEFT_BEFORE
        )

    if right_bottle is not None:
        draw_one_bottle(
            out, right_bottle,
            role_label="FIRST / RIGHT bottle",
            target_line_name="RIGHT line",
            passed_flag=right_passed,
            target_line_points=(RIGHT_LINE_P1, RIGHT_LINE_P2),
            before_color=COLOR_RIGHT_BEFORE
        )

    status_text = f"STATE: {state} | cycle={cycle_id}"
    draw_text_box(out, status_text, (15, 30), color=(255, 255, 255), bg=(0, 0, 0), scale=0.65)

    passed_text = f"LEFT passed={left_passed} | RIGHT passed={right_passed}"
    draw_text_box(out, passed_text, (15, 65), color=(255, 255, 255), bg=(0, 0, 0), scale=0.55)

    if left_dist is not None and right_dist is not None:
        dist_text = f"LEFT dist={left_dist:.1f}px | RIGHT dist={right_dist:.1f}px | tol={LINE_TOLERANCE_PX}px"
        draw_text_box(out, dist_text, (15, 98), color=(255, 255, 255), bg=(0, 0, 0), scale=0.50)

    if exact_both:
        draw_text_box(out, "BOTH CONDITIONS TRUE (&&)", (15, 132), color=(0, 255, 0), bg=(0, 0, 0), scale=0.65)

    if event_countdown > 0 and event_banner:
        draw_text_box(out, event_banner, (15, 170), color=(0, 255, 255), bg=(0, 0, 0), scale=0.65)

    return out

## 8. Process the full video with multiple cycles

This is the main cell.

It creates:
- entrance frames
- exit frames
- full overlay video
- summary tables

In [ ]:
def process_video_multi_cycles(video_path):
    info = get_video_info(video_path)
    fps = info["fps"]
    width = info["width"]
    height = info["height"]

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(OVERLAY_VIDEO_PATH), fourcc, fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer: {OVERLAY_VIDEO_PATH}")

    cap = open_video(video_path)

    entry_roi_rect = None
    detect_roi_rect = None

    # Build background from warmup frames
    frames_gray = []
    warmup_cache = []
    for i in range(BACKGROUND_WARMUP_FRAMES):
        ret, frame = cap.read()
        if not ret:
            break

        warmup_cache.append(frame.copy())
        if entry_roi_rect is None:
            entry_roi_rect = roi_from_fractions(frame.shape, ENTRY_ROI_FRACTIONS)
            detect_roi_rect = roi_from_fractions(frame.shape, DETECT_ROI_FRACTIONS)

        roi = crop_roi(frame, entry_roi_rect)
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)
        frames_gray.append(gray)

        # Write warmup frames too
        overlay = draw_overlay(
            frame=frame,
            state="WARMUP",
            cycle_id=0,
            event_banner="BUILDING BACKGROUND",
            event_countdown=1,
            entry_roi_rect=entry_roi_rect,
            detect_roi_rect=detect_roi_rect,
            left_bottle=None,
            right_bottle=None,
            left_passed=False,
            right_passed=False,
            left_dist=None,
            right_dist=None,
            exact_both=False
        )
        writer.write(overlay)

    if len(frames_gray) == 0:
        cap.release()
        writer.release()
        raise RuntimeError("Could not build background.")

    entry_background = np.median(np.stack(frames_gray, axis=0), axis=0).astype(np.uint8)

    state = "WAIT_FOR_ENTRY"
    cycle_id = 0
    event_banner = ""
    event_countdown = 0

    entry_positive_count = 0
    no_detection_count = 0

    left_passed = False
    right_passed = False
    both_frame_saved = False

    cycle_rows = []
    event_rows = []

    current_cycle = {
        "entry_frame": None,
        "exit_frame": None,
        "left_pass_frame": None,
        "right_pass_frame": None,
    }

    frame_idx = BACKGROUND_WARMUP_FRAMES

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # -----------------------------
        # OpenCV entry detection
        # -----------------------------
        entry_res = entry_foreground(frame, entry_background, entry_roi_rect)

        # -----------------------------
        # YOLO NCNN detection
        # -----------------------------
        detections, detect_roi_rect, _ = detect_bottom_half(frame)
        left_bottle, right_bottle = choose_pair(detections)

        left_dist = None
        right_dist = None
        exact_both = False

        # -----------------------------
        # State machine
        # -----------------------------
        if state == "WAIT_FOR_ENTRY":
            if entry_res["present"]:
                entry_positive_count += 1
            else:
                entry_positive_count = 0

            if entry_positive_count >= ENTRY_CONSECUTIVE_FRAMES_REQUIRED:
                cycle_id += 1
                state = "CAPTURE_WITH_LINES"
                event_banner = f"BOTTLE ENTERED | cycle {cycle_id}"
                event_countdown = ENTRY_BANNER_FRAMES

                left_passed = False
                right_passed = False
                both_frame_saved = False
                no_detection_count = 0

                current_cycle = {
                    "cycle_id": cycle_id,
                    "entry_frame": frame_idx - ENTRY_CONSECUTIVE_FRAMES_REQUIRED + 1,
                    "exit_frame": None,
                    "left_pass_frame": None,
                    "right_pass_frame": None,
                }

                # Save entrance frame
                entrance_frame = load_frame(video_path, current_cycle["entry_frame"])
                entrance_overlay = draw_overlay(
                    frame=entrance_frame,
                    state="BOTTLE_ENTERED",
                    cycle_id=cycle_id,
                    event_banner=event_banner,
                    event_countdown=1,
                    entry_roi_rect=entry_roi_rect,
                    detect_roi_rect=detect_roi_rect,
                    left_bottle=None,
                    right_bottle=None,
                    left_passed=False,
                    right_passed=False,
                    left_dist=None,
                    right_dist=None,
                    exact_both=False
                )
                entrance_path = ENTRANCE_FRAMES_DIR / f"cycle_{cycle_id:03d}_entrance_frame_{current_cycle['entry_frame']}.jpg"
                cv2.imwrite(str(entrance_path), entrance_overlay)

                event_rows.append({
                    "cycle_id": cycle_id,
                    "event": "bottle_entered",
                    "frame": current_cycle["entry_frame"],
                    "path": str(entrance_path)
                })

                entry_positive_count = 0

        elif state == "CAPTURE_WITH_LINES":
            if left_bottle is not None:
                left_reached, left_dist, _ = reached_or_passed_line(
                    (left_bottle["x_center"], left_bottle["y_center"]),
                    LEFT_LINE_P1, LEFT_LINE_P2,
                    LINE_TOLERANCE_PX
                )
                if left_reached and not left_passed:
                    left_passed = True
                    current_cycle["left_pass_frame"] = frame_idx
                    event_rows.append({
                        "cycle_id": cycle_id,
                        "event": "second_left_bottle_passed_left_line",
                        "frame": frame_idx,
                        "path": ""
                    })

            if right_bottle is not None:
                right_reached, right_dist, _ = reached_or_passed_line(
                    (right_bottle["x_center"], right_bottle["y_center"]),
                    RIGHT_LINE_P1, RIGHT_LINE_P2,
                    LINE_TOLERANCE_PX
                )
                if right_reached and not right_passed:
                    right_passed = True
                    current_cycle["right_pass_frame"] = frame_idx
                    event_rows.append({
                        "cycle_id": cycle_id,
                        "event": "first_right_bottle_passed_right_line",
                        "frame": frame_idx,
                        "path": ""
                    })

            if left_passed and right_passed:
                exact_both = True

                if not both_frame_saved:
                    current_cycle["exit_frame"] = frame_idx
                    both_frame_saved = True

                    event_banner = f"BOTH PASSED (&&) | cycle {cycle_id}"
                    event_countdown = PASS_BANNER_FRAMES

                    exit_overlay = draw_overlay(
                        frame=frame,
                        state="CAPTURE_WITH_LINES",
                        cycle_id=cycle_id,
                        event_banner=event_banner,
                        event_countdown=1,
                        entry_roi_rect=entry_roi_rect,
                        detect_roi_rect=detect_roi_rect,
                        left_bottle=left_bottle,
                        right_bottle=right_bottle,
                        left_passed=left_passed,
                        right_passed=right_passed,
                        left_dist=left_dist,
                        right_dist=right_dist,
                        exact_both=True
                    )
                    exit_path = EXIT_FRAMES_DIR / f"cycle_{cycle_id:03d}_exit_frame_{frame_idx}.jpg"
                    cv2.imwrite(str(exit_path), exit_overlay)

                    event_rows.append({
                        "cycle_id": cycle_id,
                        "event": "both_passed_and_exit_frame_saved",
                        "frame": frame_idx,
                        "path": str(exit_path)
                    })

            # After both passed, wait until the pair is gone to close the cycle.
            if left_passed and right_passed:
                if len(detections) == 0:
                    no_detection_count += 1
                else:
                    no_detection_count = 0

                if no_detection_count >= NO_DETECTION_EXIT_FRAMES:
                    state = "WAIT_FOR_ENTRY"
                    event_banner = f"BOTTLES EXITED | cycle {cycle_id}"
                    event_countdown = EXIT_BANNER_FRAMES

                    cycle_rows.append({
                        "cycle_id": cycle_id,
                        "entry_frame": current_cycle["entry_frame"],
                        "left_pass_frame": current_cycle["left_pass_frame"],
                        "right_pass_frame": current_cycle["right_pass_frame"],
                        "exit_frame": current_cycle["exit_frame"],
                    })

                    event_rows.append({
                        "cycle_id": cycle_id,
                        "event": "bottles_exited",
                        "frame": frame_idx,
                        "path": ""
                    })

                    no_detection_count = 0

        # -----------------------------
        # Draw overlay and write frame
        # -----------------------------
        overlay = draw_overlay(
            frame=frame,
            state=state,
            cycle_id=cycle_id,
            event_banner=event_banner,
            event_countdown=event_countdown,
            entry_roi_rect=entry_roi_rect,
            detect_roi_rect=detect_roi_rect,
            left_bottle=left_bottle if state == "CAPTURE_WITH_LINES" else None,
            right_bottle=right_bottle if state == "CAPTURE_WITH_LINES" else None,
            left_passed=left_passed if state == "CAPTURE_WITH_LINES" else False,
            right_passed=right_passed if state == "CAPTURE_WITH_LINES" else False,
            left_dist=left_dist if state == "CAPTURE_WITH_LINES" else None,
            right_dist=right_dist if state == "CAPTURE_WITH_LINES" else None,
            exact_both=exact_both if state == "CAPTURE_WITH_LINES" else False
        )

        writer.write(overlay)

        if event_countdown > 0:
            event_countdown -= 1

        frame_idx += 1

    cap.release()
    writer.release()

    summary_df = pd.DataFrame(cycle_rows)
    events_df = pd.DataFrame(event_rows)

    summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
    events_df.to_csv(EVENTS_CSV_PATH, index=False)

    return summary_df, events_df


summary_df, events_df = process_video_multi_cycles(VIDEO_PATH)

print("Saved overlay video:", OVERLAY_VIDEO_PATH)
print("Saved cycle summary:", SUMMARY_CSV_PATH)
print("Saved event log:", EVENTS_CSV_PATH)

display(summary_df)
display(events_df)

## 9. Show entrance and exit frames

In [ ]:
def show_image_grid(image_paths, title, ncols=2, figsize=(14, 8)):
    image_paths = [Path(p) for p in image_paths if Path(p).exists()]
    if not image_paths:
        print(f"No images found for: {title}")
        return

    n = len(image_paths)
    nrows = int(np.ceil(n / ncols))

    plt.figure(figsize=figsize)
    for i, p in enumerate(image_paths, 1):
        img = cv2.imread(str(p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(nrows, ncols, i)
        plt.imshow(img)
        plt.title(p.name)
        plt.axis("off")
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


entrance_paths = events_df.loc[events_df["event"] == "bottle_entered", "path"].tolist()
exit_paths = events_df.loc[events_df["event"] == "both_passed_and_exit_frame_saved", "path"].tolist()

show_image_grid(entrance_paths, "Entrance frames", ncols=2, figsize=(16, 8))
show_image_grid(exit_paths, "Exit frames (both conditions true &&)", ncols=2, figsize=(16, 8))

## 10. Quick preview from the saved overlay video

This cell shows a few frames from the generated overlay video.

In [ ]:
def sample_frames_from_video(video_path, frame_indices):
    cap = open_video(video_path)
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append((idx, frame))
    cap.release()
    return frames


if len(summary_df):
    sample_indices = []
    for _, row in summary_df.iterrows():
        for key in ["entry_frame", "left_pass_frame", "right_pass_frame", "exit_frame"]:
            if pd.notna(row[key]):
                sample_indices.append(int(row[key]))
    sample_indices = sorted(set(sample_indices))[:8]

    sampled = sample_frames_from_video(OVERLAY_VIDEO_PATH, sample_indices)

    plt.figure(figsize=(16, 10))
    for i, (idx, frame) in enumerate(sampled, 1):
        plt.subplot(int(np.ceil(len(sampled) / 2)), 2, i)
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f"Overlay video frame {idx}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No cycles detected yet.")

## 11. Notes / tuning

### If entry is detected too early
Increase one or more:
- `ENTRY_DIFF_THRESHOLD`
- `ENTRY_MIN_FOREGROUND_PIXELS`
- `ENTRY_MIN_LARGEST_CONTOUR_AREA`
- `ENTRY_CONSECUTIVE_FRAMES_REQUIRED`

### If YOLO NCNN is not finding the right class
Check:
```python
print(model.names)
```
Then adjust:
- `TARGET_CLASS_NAMES`
- `TARGET_CLASS_IDS`
- `YOLO_CONF`

### If the chosen line frame is wrong
Tune:
- `LEFT_LINE_P1`, `LEFT_LINE_P2`
- `RIGHT_LINE_P1`, `RIGHT_LINE_P2`
- `LINE_TOLERANCE_PX`

### Output files
- Full overlay video:
  - `overlay_video.mp4`
- Entrance frames:
  - `entrance_frames/`
- Exit frames:
  - `exit_frames/`
- Cycle summary:
  - `cycle_summary.csv`
- Event log:
  - `event_log.csv`